# Acceso a API y Obtención de Datos Climáticos
Dado que se quiere realizar una proyección de los valores de DALYs en función de su correlación con las variables socioeconómicas y ambientales. De esta manera, se accederá a la API de Climate Impact Explorer (https://climate-impact-explorer.climateanalytics.org/) para tener datos disponibles sobre impactos climáticos y su severidad a través de varios países bajo varios escenarios de cambio global. 

## Descarga de Datos por API de CIE
Se eligieron los 10 países con mayor y menor carga de DALYs para hacer dichas extrapolaciones. 
- Top 10 países: ['Nigeria', 'India', 'Niger', 'Mozambique', 'China', 'Uganda', 'Ethiopia', 'Mali', 'Burkina Faso', 'Cameroon']
- Bottom 10 países (con DALYs > 0): ['Iceland', 'Luxembourg', 'Finland', 'Bahrain', 'Denmark', 'Grenada', 'Cyprus', 'Ireland', 'Belgium', 'Qatar']


In [70]:
# Configuración de librerías. 

import pandas as pd
import numpy as np
from joblib import load
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from joblib import dump
import matplotlib.pyplot as plt

In [73]:
in_file = "data_clean/DALYs_clean_unificado_NEW.pkl"
with open(in_file, 'rb') as f:
     df_dalys= load(f)
display(df_dalys)
print(df_dalys.columns.tolist())

df_clima = pd.read_csv('data_clean/linear_model_daly_prediction/datos_climaticos_CIE.csv')

,ano,pais,codigo_pais,codigo_ghe,categoria_principal,categoria_nivel1,categoria_nivel2,causa,sexo,edad,...,esperanza_vida,poblacion_miles,co2_anual,precipitacion_total,temperatura_superficial,humedad_relativa,poblacion_abs,tasa_dalys_100k,dalys_por_intervencion,desarrollo_vs_carga
0,2000,Afghanistan,AFG,210.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,NaN,Males,0-4,...,53.76,2152.578,511.1,0.009,9.687,40.39,2152578.0,1.700371e+00,1.746827e-05,NaN
1,2000,Afghanistan,AFG,220.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,a.Malaria,Males,0-4,...,53.76,2152.578,511.1,0.009,9.687,40.39,2152578.0,5.747056e-01,6.211615e-06,NaN
2,2000,Afghanistan,AFG,230.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,b.Trypanosomiasis,Males,0-4,...,53.76,2152.578,511.1,0.009,9.687,40.39,2152578.0,0.000000e+00,4.645592e-07,NaN
3,2000,Afghanistan,AFG,240.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,c.Chagas disease,Males,0-4,...,53.76,2152.578,511.1,0.009,9.687,40.39,2152578.0,0.000000e+00,4.645592e-07,NaN
4,2000,Afghanistan,AFG,250.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,9.Parasitic and vector diseases,d.Schistosomiasis,Males,0-4,...,53.76,2152.578,511.1,0.009,9.687,40.39,2152578.0,0.000000e+00,4.645592e-07,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310795,2021,Zimbabwe,ZWE,340.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,a.Ascariasis,Females,70+,...,62.05,193.152,26239.0,0.026,21.090,57.65,193152.0,2.124487e-03,5.198515e-06,1.142974e-03
310796,2021,Zimbabwe,ZWE,350.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,b.Trichuriasis,Females,70+,...,62.05,193.152,26239.0,0.026,21.090,57.65,193152.0,1.551167e-09,5.177270e-06,8.345278e-10
310797,2021,Zimbabwe,ZWE,360.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,c.Hookworm disease,Females,70+,...,62.05,193.152,26239.0,0.026,21.090,57.65,193152.0,6.356956e-02,5.812965e-06,3.420042e-02
310798,2021,Zimbabwe,ZWE,362.0,"I.Communicable, maternal, perinatal and nutrit...",A.Infectious and parasitic diseases,10.Intestinal nematode infections,d.Food-borne trematodes,Females,70+,...,62.05,193.152,26239.0,0.026,21.090,57.65,193152.0,0.000000e+00,5.177270e-06,0.000000e+00


['ano', 'pais', 'codigo_pais', 'codigo_ghe', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 'sexo', 'edad', 'dalys', 'pib_per_capita', 'idh', 'indice_educacion', 'indice_salud', 'indice_ingresos', 'esperanza_vida', 'poblacion_miles', 'co2_anual', 'precipitacion_total', 'temperatura_superficial', 'humedad_relativa', 'poblacion_abs', 'tasa_dalys_100k', 'dalys_por_intervencion', 'desarrollo_vs_carga']


In [ ]:
# Acceso a datos

## Configuración de consulta ##

# Países seleccionados
PAIS = [
    # África (alta carga de NTDs)
    "NGA",  # Nigeria
    "COD",  # Congo Democrático
    "ETH",  # Etiopía
    "TZA",  # Tanzania
    "UGA",  # Uganda
    "KEN",  # Kenia
    "MOZ",  # Mozambique
    "MWI",  # Malawi
    "ZMB",  # Zambia
    "CMR",  # Camerún
    "NER",  # Níger
    "MLI",  # Malí
    "BFA",  # Burkina Faso
    "SEN",  # Senegal
    "GHA",  # Ghana
    
    # Asia
    "IND",  # India
    "BGD",  # Bangladesh
    "PAK",  # Pakistán
    "MMR",  # Myanmar
    "IDN",  # Indonesia
    "CHN",  # China
    
    # América Latina
    "BRA",  # Brasil
    "MEX",  # México
    "COL",  # Colombia
    "PER",  # Perú
    
    # Referencia (baja carga)
    "ISL",  # Islandia
    "LUX",  # Luxemburgo
    "FIN",  # Finlandia
    "BHR",  # Baréin
    "DNK",  # Dinamarca
    "GRD",  # Granada
    "CYP",  # Chipre
    "IRL",  # Irlanda
    "BEL",  # Bélgica
    "QAT"   # Catar
]

# Variables climáticas
VARIABLE = [
    "tasAdjust",    # Temperatura media
    "prAdjust",     # Precipitación
    "hursAdjust",   # Humedad relativa
    "tasmaxAdjust", # Temperatura diaria máxima
    "tasminAdjust"  # Temperatura diaria mínima
]
# Escenario de emisiones (usaremos el de altas emisiones 'h_cpol' y el de mitigación 'o_1p5c')
ESCENARIO = ["h_cpol" , "o_1p5c"] 

BASE_URL = "https://cie-api-v2.climateanalytics.org/api/timeseries/"

# --- Función de descarga (CORREGIDA) ---
def descargar_datos(pais, variable, escenario):
    """
    Descarga los datos de la API para una combinación específica.
    Retorna un DataFrame con los datos o None si hay error.
    """
    
    # Parámetros de la consulta
    params = {
        "iso": pais,
        "region": pais,
        "scenario": escenario,
        "var": variable,
        "aggregation_spatial": "area",
        "season": "annual",
        "format": "json"
    }
    
    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        datos = response.json()
        
        if 'status' in datos:
            return None
            
        df = pd.DataFrame({
            'year': datos['year'],
            'value': datos['median'],
            'value_lower': datos['lower'],
            'value_upper': datos['upper'],
            'warming_level': datos['warming_levels']
        })
        
        df['pais'] = pais
        df['variable'] = variable
        df['escenario'] = escenario
        
        return df
        
    except:
        return None

# --- Descarga masiva ---
print(f"Descargando {len(PAIS) * len(VARIABLE) * len(ESCENARIO)} combinaciones...")

todos = []
for pais in tqdm(PAIS):
    for var in VARIABLE:
        for esc in ESCENARIO:
            df = descargar_datos(pais, var, esc)
            if df is not None:
                todos.append(df)
            time.sleep(0.1)

# --- Guardar resultados ---
if todos:
    df_final = pd.concat(todos, ignore_index=True)
    
    # Reordenar columnas para formato tidy estándar
    columnas_orden = ['year', 'pais', 'escenario', 'variable', 'value', 'value_lower', 'value_upper', 'warming_level']
    df_final = df_final[columnas_orden]
    df_final = df_final.sort_values(['pais', 'escenario', 'variable', 'year'])
    
    df_final.to_csv('data_clean/linear_model_daly_prediction/datos_climaticos_CIE.csv', index=False)
    
    print(f"\n Completado! {len(df_final)} registros guardados en 'data_clean/linear_model_daly_prediction/datos_climaticos_CIE.csv'")
    print(df_final.head(10))
    
    # Mostrar resumen
    print("\n Resumen por variable:")
    print(df_final.groupby('variable').size())
else:
    print("No se descargaron datos. Verifica tu conexión a internet.")

Descargando 350 combinaciones...


100%|██████████| 35/35 [03:50<00:00,  6.57s/it]


 Completado! 6300 registros guardados en 'data_clean/linear_model_daly_prediction/datos_climaticos_CIE.csv'
        year pais escenario    variable      value  value_lower  value_upper  \
6012  2015.0  BEL    h_cpol  hursAdjust  80.654830    79.835887    81.305855   
6013  2020.0  BEL    h_cpol  hursAdjust  80.568827    79.682841    81.410657   
6014  2025.0  BEL    h_cpol  hursAdjust  80.446939    79.339350    81.398392   
6015  2030.0  BEL    h_cpol  hursAdjust  80.292301    79.095014    81.370471   
6016  2035.0  BEL    h_cpol  hursAdjust  80.158094    78.996112    81.335199   
6017  2040.0  BEL    h_cpol  hursAdjust  80.048425    78.826478    81.327597   
6018  2045.0  BEL    h_cpol  hursAdjust  79.961415    78.551974    81.330905   
6019  2050.0  BEL    h_cpol  hursAdjust  79.884367    78.348219    81.356348   
6020  2055.0  BEL    h_cpol  hursAdjust  79.842134    78.207203    81.384475   
6021  2060.0  BEL    h_cpol  hursAdjust  79.807045    78.121132    81.418485   

      warm

## Entrenamiento con años comunes entre los DataFrames
Se entrenará un modelo lineal con los años de 2015 y 2020. Para ello se van a concatenar el DataFrame de DALYs y el de datos climáticos dados por la API.